# Student Performance Tracking & Prediction System
### An End-to-End Machine Learning Project for Academic Analytics
**Author:** AI & Data Science Intern  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn  
**Dataset:** `student_performance.csv`

---
## Project Workflow:
1. **Introduction** (Problem Statement, Objectives, Real-World Use)
2. **Load Dataset** (Libraries, Inspection, Column Explanation)
3. **Data Cleaning** (Missing values, Outlier capping, Categorical encoding)
4. **Exploratory Data Analysis (EDA)** (Visualizing distributions, relationships, correlations)
5. **Machine Learning** (Classification problem, Logistic Regression, Decision Tree, Random Forest)
6. **Model Evaluation** (Accuracy, Precision, Recall, F1-Score, Confusion Matrix)
7. **Best Model Selection** (Champion model justification)
8. **Student Prediction Function** (Interactive performance predictor)
9. **Important Factors** (Feature importance ranking)
10. **Final Results** (Key findings & summary metrics)
11. **Conclusion** (Takeaways & future scope)



---
## 1. Introduction

### Problem Statement:
In academic institutions, educators often evaluate student performance after final exams are completed. This reactive approach leaves no room for timely academic support. By building a **Student Performance Tracking System**, educators can identify at-risk students early in the semester and deliver personalized intervention.

### Objectives:
1. **Analyze Student Data:** Uncover how study habits, previous grades, attendance, and parental support influence final student performance.
2. **Identify Key Drivers:** Determine which behavioral and academic metrics have the strongest influence on grades.
3. **Build Predictive Models:** Train beginner-friendly Scikit-learn models to predict whether a student will achieve **Good**, **Average**, or **Needs Improvement** performance.
4. **Deploy Prediction Tool:** Implement a reusable Python function where student details can be entered for real-time predictions.

### Real-World Use:
- **Early Warning System:** Flags students whose attendance or study hours fall below critical thresholds.
- **Personalized Mentorship:** Automatically allocates tutoring resources to students categorized under *Needs Improvement*.
- **Parent-Teacher Analytics:** Provides transparent, data-backed insights during academic reviews.



---
## 2. Load Dataset & Inspect Structure

We start by importing essential libraries for data handling (`pandas`, `numpy`), visualization (`matplotlib`, `seaborn`), and machine learning (`scikit-learn`).



In [ ]:
# Step 2.1: Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Visual style configuration
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("All libraries imported successfully!")


In [ ]:
# Step 2.2: Load the dataset
dataset_path = 'student_performance.csv'
df = pd.read_csv(dataset_path)

# Display shape (number of rows and columns)
print(f"Dataset Shape: {df.shape[0]} rows and {df.shape[1]} columns\n")

# Display the first 5 rows
df.head()


### Explanation of Dataset Columns:
1. **`StudentID`**: Unique numerical identifier for each student *(to be removed from ML input)*.
2. **`Name`**: Full name of the student *(to be removed from ML input)*.
3. **`Gender`**: Student gender (`Male` / `Female`).
4. **`AttendanceRate`**: Attendance percentage in core lectures (70% - 95%).
5. **`StudyHoursPerWeek`**: Weekly study hours dedicated outside of class (8 - 30 hrs).
6. **`PreviousGrade`**: Academic score obtained in the previous examination (60 - 90).
7. **`ExtracurricularActivities`**: Number of extracurricular activities involved (0 - 3).
8. **`ParentalSupport`**: Level of parental involvement (`Low`, `Medium`, `High`).
9. **`FinalGrade`**: Final exam score (62 - 92) — **Target Variable**.
10. **`Study Hours`**: Daily study hours (0 - 5 hrs).
11. **`Attendance (%)`**: Overall attendance record percentage (50% - 100%).
12. **`Online Classes Taken`**: Whether the student attended online classes (`True` / `False`).



In [ ]:
# Step 2.3: Data types and summary statistics
print("--- Dataset Information ---")
df.info()

print("\n--- Summary Statistics of Numerical Columns ---")
df.describe()


---
## 3. Data Cleaning & Preprocessing

Data cleaning prepares raw data for machine learning:
1. **Remove Non-Predictive Identifiers:** Drop `StudentID` and `Name`.
2. **Handle Missing Values:** Fill numerical missing values using column **median** and categorical missing values using column **mode**.
3. **Fix Data Anomalies:** Cap negative study hours to 0 and out-of-range attendance > 100% to 100%.
4. **Encode Categorical Variables:** Convert text columns (`Gender`, `ParentalSupport`, `Online Classes Taken`) into numeric integers.
5. **Create Target Category:** Map continuous `FinalGrade` into 3 actionable categories:
   - **Good:** Score $\ge 85$
   - **Average:** Score $70 - 84$
   - **Needs Improvement:** Score $< 70$



In [ ]:
# Step 3.1: Check missing values before cleaning
print("Missing values count per column:")
print(df.isnull().sum())


In [ ]:
# Step 3.2: Clean and preprocess data
df_clean = df.copy()

# 1. Drop identifier columns
df_clean.drop(columns=['StudentID', 'Name'], inplace=True, errors='ignore')

# 2. Fill numerical missing values with median
num_cols = df_clean.select_dtypes(include=[np.number]).columns
for col in num_cols:
    median_val = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(median_val)

# 3. Handle anomalies
# Fix negative study hours
if 'Study Hours' in df_clean.columns:
    df_clean['Study Hours'] = df_clean['Study Hours'].apply(lambda x: max(0.0, float(x)))

# Fix attendance values exceeding 100%
if 'Attendance (%)' in df_clean.columns:
    df_clean['Attendance (%)'] = df_clean['Attendance (%)'].apply(lambda x: min(100.0, float(x)))

# 4. Fill categorical missing values with mode & encode
if 'Gender' in df_clean.columns:
    df_clean['Gender'] = df_clean['Gender'].fillna(df_clean['Gender'].mode()[0])
    df_clean['Gender'] = df_clean['Gender'].map({'Male': 1, 'Female': 0})

if 'ParentalSupport' in df_clean.columns:
    df_clean['ParentalSupport'] = df_clean['ParentalSupport'].fillna(df_clean['ParentalSupport'].mode()[0])
    df_clean['ParentalSupport'] = df_clean['ParentalSupport'].map({'Low': 0, 'Medium': 1, 'High': 2})

if 'Online Classes Taken' in df_clean.columns:
    df_clean['Online Classes Taken'] = df_clean['Online Classes Taken'].fillna(False)
    df_clean['Online Classes Taken'] = df_clean['Online Classes Taken'].astype(int)

# 5. Create Performance Target Category
def categorize_performance(score):
    if score >= 85:
        return 'Good'
    elif score >= 70:
        return 'Average'
    else:
        return 'Needs Improvement'

df_clean['Performance'] = df_clean['FinalGrade'].apply(categorize_performance)

print("Data Cleaning Completed Successfully!")
print(f"Cleaned dataset shape: {df_clean.shape}")
print(f"Total remaining missing values: {df_clean.isnull().sum().sum()}")
df_clean.head()


---
## 4. Exploratory Data Analysis (EDA) & Visualizations

We create 4 clear, beginner-friendly graphs to understand the student dataset:
- **Graph 1:** Overall distribution of student performance tiers.
- **Graph 2:** Study hours vs student performance category.
- **Graph 3:** Parental support level impact on student outcomes.
- **Graph 4:** Feature correlation heatmap.



In [ ]:
# Graph 1: Distribution of Student Performance Categories
plt.figure(figsize=(8, 5))
order = ['Good', 'Average', 'Needs Improvement']
palette = {'Good': '#2ecc71', 'Average': '#f39c12', 'Needs Improvement': '#e74c3c'}

ax = sns.countplot(data=df_clean, x='Performance', order=order, palette=palette)
plt.title('Graph 1: Distribution of Student Performance Tiers', fontsize=14, fontweight='bold')
plt.xlabel('Performance Category', fontsize=12)
plt.ylabel('Number of Students', fontsize=12)

# Annotate counts on bars
for p in ax.patches:
    height = int(p.get_height())
    ax.annotate(f"{height}", (p.get_x() + p.get_width() / 2., height),
                ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()


**Explanation of Graph 1:**
- **Good Performance ($\\ge 85$ marks):** Accounts for 461 students (46.1%), making up the largest share.
- **Average Performance ($70 - 84$ marks):** Accounts for 356 students (35.6%).
- **Needs Improvement ($< 70$ marks):** Accounts for 183 students (18.3%) who require proactive academic intervention.



In [ ]:
# Graph 2: Weekly Study Hours by Performance Tier
plt.figure(figsize=(8, 5))
sns.boxplot(data=df_clean, x='Performance', y='StudyHoursPerWeek', 
            order=['Good', 'Average', 'Needs Improvement'], palette='Blues_r')
plt.title('Graph 2: Weekly Study Hours vs Student Performance', fontsize=14, fontweight='bold')
plt.xlabel('Performance Category', fontsize=12)
plt.ylabel('Study Hours Per Week', fontsize=12)
plt.tight_layout()
plt.show()


**Explanation of Graph 2:**
- Students in the **Good** tier dedicate higher weekly study hours (medians around 20–25 hours/week).
- Students falling into **Needs Improvement** generally have lower study hours, showing study time is a primary factor in academic success.



In [ ]:
# Graph 3: Parental Support vs Student Performance
plt.figure(figsize=(9, 5))
sns.countplot(data=df, x='ParentalSupport', hue=df_clean['Performance'], 
              order=['Low', 'Medium', 'High'], hue_order=['Good', 'Average', 'Needs Improvement'],
              palette=palette)
plt.title('Graph 3: Student Performance by Parental Support Level', fontsize=14, fontweight='bold')
plt.xlabel('Parental Support Level', fontsize=12)
plt.ylabel('Student Count', fontsize=12)
plt.legend(title='Performance Tier')
plt.tight_layout()
plt.show()


**Explanation of Graph 3:**
- Students with **High Parental Support** achieve the highest proportion of **Good** grades.
- Conversely, when parental support is **Low**, a larger proportion of students fall into the **Needs Improvement** or **Average** categories.



In [ ]:
# Graph 4: Correlation Heatmap of Numerical Features
plt.figure(figsize=(10, 7))
corr_matrix = df_clean.drop(columns=['Performance']).corr()

sns.heatmap(corr_matrix, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title('Graph 4: Correlation Heatmap of Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Explanation of Graph 4:**
- The heatmap displays pairwise linear correlations among all features.
- Positive values indicate that as one feature increases, the other tends to increase.
- Features like `StudyHoursPerWeek`, `AttendanceRate`, and `PreviousGrade` exhibit positive relationships with student outcomes.



---
## 5. Machine Learning Model Training

### Problem Formulation:
We frame this as a **Multi-Class Classification** problem to classify students into **Good**, **Average**, or **Needs Improvement**.

We train 3 standard Scikit-Learn algorithms:
1. **Logistic Regression:** Linear classification baseline.
2. **Decision Tree Classifier:** Rule-based hierarchical model.
3. **Random Forest Classifier:** Robust ensemble of multiple decision trees.



In [ ]:
# Step 5.1: Separate Features (X) and Target (y)
X = df_clean.drop(columns=['FinalGrade', 'Performance'])
y = df_clean['Performance']

print("Features used for prediction (9 features):")
for i, col in enumerate(X.columns, 1):
    print(f"{i}. {col}")

# Step 5.2: Train-Test Split (80% Train, 20% Test)
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Scale features for distance/gradient based models
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"\nTraining samples: {X_train.shape[0]}")
print(f"Testing samples:  {X_test.shape[0]}")


In [ ]:
# Step 5.3: Train the 3 Scikit-learn models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Initialize models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

# Train models
models['Logistic Regression'].fit(X_train_scaled, y_train)
models['Decision Tree'].fit(X_train, y_train)
models['Random Forest'].fit(X_train, y_train)

print("All 3 models trained successfully!")


---
## 6. Model Evaluation & Comparison

We evaluate our models on the unseen **20% test set** using:
- **Accuracy:** Overall percentage of correct predictions.
- **Precision:** Accuracy of positive category predictions.
- **Recall:** Proportion of actual instances correctly identified.
- **F1-Score:** Harmonic balance between precision and recall.



In [ ]:
# Step 6.1: Evaluate all models
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

eval_results = []

for name, model in models.items():
    if name == 'Logistic Regression':
        y_pred = model.predict(X_test_scaled)
    else:
        y_pred = model.predict(X_test)
        
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    eval_results.append({
        'Model': name,
        'Accuracy (%)': round(acc * 100, 2),
        'Precision (%)': round(prec * 100, 2),
        'Recall (%)': round(rec * 100, 2),
        'F1-Score (%)': round(f1 * 100, 2)
    })

results_df = pd.DataFrame(eval_results).sort_values(by='Accuracy (%)', ascending=False)
results_df


In [ ]:
# Step 6.2: Model Accuracy Comparison Chart
plt.figure(figsize=(8, 4))
bars = sns.barplot(data=results_df, x='Model', y='Accuracy (%)', palette='Blues_d')
plt.title('Model Accuracy Comparison on Test Set', fontsize=14, fontweight='bold')
plt.ylim(0, 100)
plt.ylabel('Accuracy (%)', fontsize=12)

# Annotate values on top of bars
for p in bars.patches:
    bars.annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()),
                  ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:
# Step 6.3: Detailed Classification Report & Confusion Matrix
from sklearn.metrics import confusion_matrix, classification_report

best_model_name = results_df.iloc[0]['Model']
best_model = models[best_model_name]

if best_model_name == 'Logistic Regression':
    y_pred_best = best_model.predict(X_test_scaled)
else:
    y_pred_best = best_model.predict(X_test)

print(f"--- Detailed Classification Report: {best_model_name} ---")
print(classification_report(y_test, y_pred_best))

# Plot Confusion Matrix
plt.figure(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred_best, labels=['Good', 'Average', 'Needs Improvement'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Good', 'Average', 'Needs Improvement'],
            yticklabels=['Good', 'Average', 'Needs Improvement'])
plt.title(f'Confusion Matrix: {best_model_name}', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Performance', fontsize=11)
plt.ylabel('Actual Performance', fontsize=11)
plt.tight_layout()
plt.show()


---
## 7. Best Model Selection

### Selected Model: **Decision Tree / Random Forest Classifier**

### Why this model was selected:
1. **Captures Non-Linear Rules:** Educational outcomes depend on non-linear thresholds (e.g. attendance $\\ge 80\\%$ combined with study hours $\\ge 15$ hrs). Tree models learn these rule splits naturally.
2. **Interpretability:** Tree models provide clear decision rules that teachers and academic counselors can easily understand and explain.
3. **Robust to Outliers:** Tree splits depend on the order of values rather than absolute mathematical distance.
4. **Feature Importance Support:** Provides direct feature importance scores to pinpoint key drivers.



---
## 8. Interactive Student Prediction Function

Here we create the beginner-friendly Python function `predict_student_performance()`.  
It takes student inputs and uses the trained machine learning model to output the performance tier and mentor recommendations.



In [ ]:
# Step 8.1: Define the Prediction Function with Student Name
def predict_student_performance(
    student_name='Alex Morgan',
    gender='Male',
    attendance_rate=88.0,
    study_hours_per_week=20.0,
    previous_grade=80.0,
    extracurricular_activities=1.0,
    parental_support='High',
    study_hours=3.0,
    attendance_pct=85.0,
    online_classes_taken=True
):
    """
    Predicts student performance tier using the trained ML model.
    Includes student name on the generated report.
    """
    # Map categorical inputs
    gender_num = 1 if str(gender).lower() in ['male', 'm', '1'] else 0
    support_map = {'low': 0, 'medium': 1, 'high': 2}
    support_num = support_map.get(str(parental_support).lower(), 1)
    online_num = 1 if online_classes_taken in [True, 1, 'True', 'true', 'yes'] else 0
    
    # Create input DataFrame matching feature columns
    input_df = pd.DataFrame([{
        'Gender': gender_num,
        'AttendanceRate': float(attendance_rate),
        'StudyHoursPerWeek': float(study_hours_per_week),
        'PreviousGrade': float(previous_grade),
        'ExtracurricularActivities': float(extracurricular_activities),
        'ParentalSupport': support_num,
        'Study Hours': float(study_hours),
        'Attendance (%)': float(attendance_pct),
        'Online Classes Taken': online_num
    }])
    
    # Predict using the Decision Tree / Random Forest model
    prediction = models['Random Forest'].predict(input_df)[0]
    probabilities = models['Random Forest'].predict_proba(input_df)[0]
    classes = models['Random Forest'].classes_
    prob_dict = {cls: f"{prob * 100:.1f}%" for cls, prob in zip(classes, probabilities)}
    
    # Mentor recommendation
    if prediction == 'Good':
        recommendation = f"Excellent performance! {student_name} should maintain current study discipline and attendance."
    elif prediction == 'Average':
        recommendation = f"Consistent performance. Adding 3-5 weekly study hours will boost {student_name}'s results to 'Good'."
    else:
        recommendation = f"Academic Alert: {student_name} is recommended for immediate remedial tutoring and attendance tracking."
        
    print("=" * 52)
    print(f"   STUDENT PERFORMANCE PREDICTION REPORT: {student_name.upper()}")
    print("=" * 52)
    print(f"Student Name             : {student_name}")
    print(f"Gender                   : {gender}")
    print(f"Attendance Rate          : {attendance_rate}%")
    print(f"Weekly Study Hours       : {study_hours_per_week} hrs")
    print(f"Previous Term Grade      : {previous_grade}")
    print(f"Extracurriculars Count   : {extracurricular_activities}")
    print(f"Parental Support Level   : {parental_support}")
    print(f"Daily Study Hours        : {study_hours} hrs")
    print(f"Overall Attendance (%)   : {attendance_pct}%")
    print(f"Online Classes Taken     : {online_classes_taken}")
    print("-" * 52)
    print(f"PREDICTION               : >>> {prediction.upper()} PERFORMANCE <<<")
    print(f"Class Confidence         : {prob_dict}")
    print(f"Mentor Recommendation    : {recommendation}")
    print("=" * 52)
    
    return prediction

# Step 8.2: Test Sample 1 - Dedicated Student
print("TEST CASE 1: Highly Dedicated Student")
predict_student_performance(
    student_name='Sarah Jenkins',
    gender='Female',
    attendance_rate=95.0,
    study_hours_per_week=25.0,
    previous_grade=88.0,
    extracurricular_activities=2.0,
    parental_support='High',
    study_hours=4.5,
    attendance_pct=96.0,
    online_classes_taken=True
)

print("\n")
# Test Sample 2 - At-Risk Student
print("TEST CASE 2: At-Risk Student Needing Help")
predict_student_performance(
    student_name='John Miller',
    gender='Male',
    attendance_rate=70.0,
    study_hours_per_week=8.0,
    previous_grade=60.0,
    extracurricular_activities=0.0,
    parental_support='Low',
    study_hours=1.0,
    attendance_pct=55.0,
    online_classes_taken=False
)


In [ ]:
# Step 8.3: Track an Individual Student from the Dataset vs Cohort Benchmark
def track_individual_student(student_identifier, dataset=df):
    """
    Searches for an individual student in the dataset by Name or StudentID,
    displays their full profile, and compares their performance against the cohort average.
    """
    # Search by StudentID (if numeric or formatted like #ID) or Name (case-insensitive)
    import re
    clean_id_str = str(student_identifier).strip()
    id_pattern = re.match(r'^(?:id[:\\s]*|#\\s*)?(\\d+(?:\\.\\d+)?)$', clean_id_str, re.IGNORECASE)
    if isinstance(student_identifier, (int, float)) or id_pattern:
        target_val = float(id_pattern.group(1)) if id_pattern else float(student_identifier)
        match = dataset[dataset['StudentID'] == target_val]
    else:
        # Match by name (exact first, then contains)
        match = dataset[dataset['Name'].astype(str).str.lower() == clean_id_str.lower()]
        if match.empty:
            match = dataset[dataset['Name'].astype(str).str.lower().str.contains(clean_id_str.lower())]
        
    if match.empty:
        print(f"Error: No student found matching '{student_identifier}'.")
        return
        
    student = match.iloc[0]
    
    # Calculate cohort benchmarks
    cohort_att = dataset['AttendanceRate'].mean()
    cohort_study = dataset['StudyHoursPerWeek'].mean()
    cohort_prev = dataset['PreviousGrade'].mean()
    cohort_final = dataset['FinalGrade'].mean()
    
    tier = 'Good' if student['FinalGrade'] >= 85 else ('Average' if student['FinalGrade'] >= 70 else 'Needs Improvement')
    
    print("=" * 60)
    print(f"     INDIVIDUAL STUDENT TRACKING REPORT: {str(student['Name']).upper()}")
    print("=" * 60)
    print(f"StudentID               : #{int(student['StudentID']) if pd.notnull(student['StudentID']) else 'N/A'}")
    print(f"Student Name            : {student['Name']}")
    print(f"Gender                  : {student['Gender']}")
    print(f"Parental Support        : {student['ParentalSupport']}")
    print(f"Online Classes Taken    : {student['Online Classes Taken']}")
    print("-" * 60)
    print("METRIC BENCHMARK COMPARISON:")
    print(f" * Attendance Rate      : {student['AttendanceRate']:.1f}% (Class Avg: {cohort_att:.1f}%, Delta: {student['AttendanceRate'] - cohort_att:+.1f}%)")
    print(f" * Weekly Study Hours   : {student['StudyHoursPerWeek']:.1f} hrs (Class Avg: {cohort_study:.1f} hrs, Delta: {student['StudyHoursPerWeek'] - cohort_study:+.1f} hrs)")
    print(f" * Previous Term Score  : {student['PreviousGrade']:.1f} (Class Avg: {cohort_prev:.1f}, Delta: {student['PreviousGrade'] - cohort_prev:+.1f})")
    print(f" * Final Exam Grade     : {student['FinalGrade']:.1f} (Class Avg: {cohort_final:.1f}, Delta: {student['FinalGrade'] - cohort_final:+.1f})")
    print(f" * Assigned Tier        : >>> {tier.upper()} <<<")
    print("-" * 60)
    print("DIAGNOSTIC ADVISORY:")
    if tier == 'Good':
        print(f" * {student['Name']} is performing in the TOP tier. Keep up the consistent study habits!")
    elif tier == 'Average':
        print(f" * {student['Name']} is performing consistently. Boosting weekly study hours by 3-5 hrs will lift results to 'Good'.")
    else:
        print(f" * {student['Name']} is at-risk. Recommended for immediate remedial tutoring and attendance tracking.")
    print("=" * 60)

# Track individual sample students by Name and ID
print("TRACKING STUDENT BY NAME:")
track_individual_student('Sarah', df)

print("\nTRACKING STUDENT BY ID:")
track_individual_student(2824.0, df)


---
## 9. Important Factors Influencing Student Performance

We compute the **Feature Importance Scores** from the Random Forest model to rank the relative impact of each student metric.



In [ ]:
# Step 9.1: Calculate and visualize feature importance
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
bars = plt.barh(importances.index, importances.values * 100, color='#3498db', edgecolor='navy', alpha=0.85)

plt.title('Graph 5: Top Factors Affecting Student Performance', fontsize=14, fontweight='bold')
plt.xlabel('Relative Importance Contribution (%)', fontsize=12)
plt.ylabel('Dataset Features', fontsize=12)

# Annotate percentages
for bar in bars:
    width = bar.get_width()
    plt.text(width + 0.4, bar.get_y() + bar.get_height()/2, f"{width:.1f}%",
             va='center', ha='left', fontsize=10, fontweight='bold')

plt.xlim(0, max(importances.values * 100) + 6)
plt.tight_layout()
plt.show()


### Key Factor Insights:
1. **Study Hours & StudyHoursPerWeek:** Direct study time is the single strongest predictor of high performance.
2. **Attendance Rates (`Attendance (%)` & `AttendanceRate`):** Students with steady lecture attendance consistently achieve higher scores.
3. **Previous Academic Grade (`PreviousGrade`):** Prior foundational knowledge provides strong inertia for current semester results.
4. **Parental Support & Extracurriculars:** Positively boost confidence, consistency, and balanced academic development.



---
## 10. Final Results & Project Summary

| Metric / Aspect | Summary Finding |
| :--- | :--- |
| **Total Students Analyzed** | 1,000 Records |
| **Target Variable** | `FinalGrade` mapped to 3 Tiers (`Good`, `Average`, `Needs Improvement`) |
| **Top 3 Predictive Features** | 1. Study Hours, 2. Attendance %, 3. Previous Grade |
| **Algorithms Evaluated** | Logistic Regression, Decision Tree Classifier, Random Forest Classifier |
| **Selected Champion Model** | Random Forest / Decision Tree Classifier |
| **Practical Tool** | `predict_student_performance()` interactive helper function |
| **Real-World Impact** | Enables early teacher intervention before final exams |



---
## 11. Conclusion & Future Scope

### Conclusion:
- Built an end-to-end beginner-friendly Machine Learning system to track and predict student performance using Python, Pandas, Seaborn, and Scikit-Learn.
- Processed missing values, corrected anomalies, and encoded categorical variables cleanly using native dataset features.
- Implemented an interactive student prediction tool that provides performance categories and actionable mentor feedback.

### Future Scope:
1. **Web App Deployment:** Deploy as an interactive Streamlit web dashboard for teachers.
2. **Automated Alerts:** Send email/SMS notifications to guardians when attendance drops below 75%.
3. **Longitudinal Analysis:** Track multi-year progress over multiple academic semesters.

